# Titanic - Machine Learning from Disaster

In [ ]:
import os

print("Các thư mục trong /kaggle/input:")
print(os.listdir('/kaggle/input'))

print("\nTìm các file CSV:")
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

## Import thư viện

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

## Load dữ liệu

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print("Train:", train_df.shape)
print("Test:", test_df.shape)

## Preview data

In [ ]:
print('Loaded:', train_df.shape, test_df.shape)
train_df.head()

In [ ]:
test_df.tail()

## Exploratory Data Analysis - EDA

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
def display_missing_data(df):
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    percent = (missing / len(df)) * 100
    print(pd.DataFrame({'Missing Values': missing, 'Percent (%)': percent.round(2)}))

display_missing_data(train_df)

In [ ]:
display_missing_data(test_df)

In [ ]:
def best_features_for_imputation(df, target='Age', top=10):
    """
    Phân tích mối tương quan giữa 1 feature (vd: Age)
    với tất cả các feature khác trong dataframe để gợi ý 
    nên dùng cột nào để điền missing values.
    ----------------------------------------------------
    df: DataFrame
    target: tên cột muốn xem tương quan (vd 'Age')
    top: số lượng feature top đầu cần hiển thị
    """
    corrs = {}
    for col in df.columns:
        if col == target:
            continue

        try:
            # Nếu là numeric -> dùng Spearman correlation
            if np.issubdtype(df[col].dropna().dtype, np.number):
                corr = spearmanr(df[target], df[col], nan_policy='omit')[0]
            else:
                # Nếu là object -> factorize (chuyển sang mã số)
                corr = spearmanr(df[target], pd.factorize(df[col])[0], nan_policy='omit')[0]
            corrs[col] = abs(corr)
        except Exception as e:
            continue

    # Sắp xếp giảm dần
    corrs_sorted = pd.Series(corrs).sort_values(ascending=False)
    result = corrs_sorted.head(top).to_frame('Spearman Corr (|r|)')
    print(f"\nTop {top} features liên quan mạnh nhất đến '{target}':\n")
    print(result)

### Target Variable: `Survived`

In [ ]:
print(train_df.shape)
print(train_df.columns.tolist())
print(train_df.head())

In [ ]:
survival_counts = train_df['Survived'].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(5,5))
plt.pie(
    survival_counts,
    labels=['Not Survived (0)', 'Survived (1)'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#ff9999','#66b3ff']
)
plt.title('Survival Distribution in Titanic Dataset')
plt.show()

Chỉ có khoảng 38.4% hành khách sống sót trong thảm họa Titanic.

- Phân bố của các Features

In [ ]:
cols = [ 'Sex', 'Embarked', 'Pclass', 'SibSp', 'Parch'] 

n_rows = 2
n_cols = 3

fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.5, n_rows*3.5))

for r in range(0, n_rows):
    for c in range(0, n_cols):
        i = r*n_cols + c
        if i < len(cols):
            ax_i = ax[r,c]
            sns.countplot(data=train_df, x=cols[i], hue='Survived', palette='Blues', ax=ax_i)
            ax_i.set_title(f'Figure {i+1}: Survival Rate for {cols[i]}')
            ax_i.legend(title='', loc='upper right', labels=['Not Survived', 'Survived'])
ax.flat[-1].set_visible(False)
plt.tight_layout()

**Tỉ lệ sống sót**:
- Fig 1: Tỉ lệ sống sót của nữ giới cao hơn nam giới.
- Fig 2: Đa phần mọi người đi đến Southampton, và cũng có lượng người không sống sót cao nhất.
- Fig 3: Vé hạng 1 có tỉ lệ sống sót cao hơn.
- Fig 4: Những người với `SibSp` bằng 0 hầu hết sẽ không sống sót, số lượng hành khách đi với 1-2 thành viên gia đình có cơ hội sống sót cao hơn.
- Fig 5: Những người với `Parch` bằng 0 đa phần sẽ không sống sót, số lượng hành khách đi với 1-2 thành viên gia đình có cơ hội sống sót cao hơn.

- Age

In [ ]:
sns.histplot(data=train_df, x='Age', hue='Survived', bins=40, kde=True)

- Đa phần hành khách có độ tuổi từ 18-40 tuổi.
- Trẻ em có nhiều cơ hội sống sót hơn những độ tuổi khác

- Fare

In [ ]:
sns.histplot(data=train_df, x='Fare', hue='Survived', bins=40)

In [ ]:
best_features_for_imputation(train_df, target='Survived', top=20)

**Ta có các biến liên quan đến `Survived` và không bị missing:**
- Sex
- Pclass
- SibSp / Parch
- Name

## Feature Engineering (Các features không bị missing)

#### Tạo thêm các biến mới giúp mô hình phân tích chuẩn xác và mạnh hơn

- `Title`: Phản ánh giới tính, địa vị, tầng lớp xã hội.
- `IsFemale`: Giới tính nữ được ưu tiên hơn.
- `FamilySize`: Quy mô gia đình ảnh hưởng tỉ lệ sống, đi cùng thành viên gia đình có tỉ lệ sống sót cao hơn.
- `IsChild`: Đánh dấu trẻ em (dưới 12 tuổi). Trong Titanic, trẻ em thường được cứu trước theo nguyên tắc “women and children first”.
- `IsMother`: Đánh dấu phụ nữ trưởng thành có con đi cùng. Mẹ thường được ưu tiên cứu cùng con.

In [ ]:
# Feature engineering
def extract_title(name):
    m = re.search(r',\s*([^\.]+)\.', name)
    return m.group(1).strip() if m else ''

title_map = {
    'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Officer',
    'Don': 'Royalty', 'Sir': 'Royalty', 'Lady': 'Royalty', 'the Countess': 'Royalty', 'Jonkheer': 'Royalty', 'Dona': 'Royalty'
}

for df in [train_df, test_df]:
    # --- Title (Name) ---
    df['Title'] = df['Name'].apply(extract_title)
    df['Title'] = df['Title'].replace(['Mlle','Ms'],'Miss')
    df['Title'] = df['Title'].replace(['Mme'],'Mrs')
    df['Title'] = df['Title'].replace(title_map)
    rare = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
    df['Title'] = df['Title'].replace(list(rare), 'Other')

    # --- Family (SibSp/Parch) ---
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # --- Special groups ---
    df['IsChild'] = (df['Age'] < 12).astype(int)
    df['IsMother'] = ((df['Sex'] == 'female') & (df['Parch'] > 0) & (df['Age'] > 18) & (df['Title'] == 'Mrs')).astype(int)

## Filling missing values

In [ ]:
display_missing_data(train_df)

- Trong file train.csv, chỉ có 2 giá trị bị thiếu (NaN) trên tổng 891 dòng —
tức là 0.2% dữ liệu → rất ít, không ảnh hưởng lớn đến phân phối.
- Chọn mode() là giá trị xuất hiện nhiều nhất trong cột để điền "giá trị đại diện" cho `Embarked`.

In [ ]:
display_missing_data(test_df)

- Fare chỉ thiếu 1 giá trị → nên điền theo median.

In [ ]:
best_features_for_imputation(train_df, target='Fare', top=10)

- Ta thấy Pclass liên quan mạnh nhất đến `Fare` nên điền missing value của `Fare` bằng median của nhóm Pclass.

Ta thấy được trong các features có liên quan đến `Age`,:
- Parch: Hạng vé, tầng lớp xã hội. Người giàu (Pclass=1) thường lớn tuổi hơn.
- Chức danh (Mr, Miss, Dr, Master...). Phản ánh tuổi rõ nhất (Mr > Master, Miss trẻ hơn Mrs).
- Giới tính. Nam/Nữ có độ tuổi trung bình khác nhau trong từng nhóm xã hội
- Khi điền giá trị thiếu (Age), ta muốn ước lượng gần đúng độ tuổi thật của hành khách, ta chọn `Sex + Pclass + Title` là nhóm tối ưu nhất

In [ ]:
# Fill missing values
for df in [train_df, test_df]:
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
    
    df['Age'] = df.groupby(['Sex','Pclass','Title'])['Age'].transform(lambda x: x.fillna(x.median()))

    # --- Chuẩn hóa log ---
    df['Fare'] = np.log1p(df['Fare'])
    df['Age'] = np.log1p(df['Age'])

## Feature Engineering (Các biến khác)

In [ ]:
for df in [train_df, test_df]:

    df['HasCabin'] = df['Cabin'].notna().astype(int)

    df['Age*Pclass'] = df['Age'] * df['Pclass']

    # --- Cabin / Deck ---
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df['Deck'] = df['Deck'].replace('n', 'U')  # 'U' = Unknown

    # --- Interaction features ---
    df['Fare_Pclass'] = df['Fare'] / df['Pclass']

    df['TicketPrefix'] = df['Ticket'].apply(lambda x: re.split(r'\s|\.', str(x))[0])
    df['TicketPrefix'] = df['TicketPrefix'].apply(lambda x: x if x.isalpha() else 'NUM')

- `HasCabin`: Có Cabin hay không. Người có cabin → hạng vé cao → tầng lớp giàu có.
- `Deck`: Boong (deck) tàu – ký tự đầu của Cabin (A, B, C, D...). Deck thấp → gần đáy tàu → nguy hiểm hơn.
- `Age*Pclass`: Kết hợp tuổi và hạng vé – phản ánh “tầng lớp xã hội theo tuổi”.
- `Fare_Pclass`: Giá vé chia theo hạng vé.
- `TicketPrefix`: Phần đầu vé cho biết hãng tàu hoặc loại vé.

### Các features để mô hình phân tích

In [ ]:
# Prepare features
feature_cols = [
    'Pclass','Sex','Age','Fare','Embarked','Title','FamilySize','IsChild','IsMother',
    'Deck','HasCabin','Fare_Pclass','Age*Pclass','TicketPrefix'
]
x = train_df[feature_cols]
y = train_df['Survived']
x_test_final = test_df[feature_cols]

### Preprocess pipeline

In [ ]:
# Preprocessor (ColumnTransformer)
num_features = [
    'Age','Fare','FamilySize','Fare_Pclass','Age*Pclass'
]
cat_features = [
    'Pclass','Sex','Embarked','Title','IsChild','IsMother',
    'Deck','HasCabin','TicketPrefix'
]

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features), ('cat', cat_transformer, cat_features)])

## Models Training

**Các mô hình dùng để phân tích**: `Logistic Regression`, `Random Forest`, `XGBoost`, `SVM`

In [ ]:
# --- Khởi tạo các pipeline ---
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000,random_state=42,C=1.0,solver='lbfgs'))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=120,max_depth=2,learning_rate=0.08,subsample=0.6,colsample_bytree=0.6,reg_lambda=3,reg_alpha=2))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

### Đánh giá từng model bằng cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)

for name, model in models.items():
    acc = cross_val_score(model, x, y, cv=cv, scoring='accuracy').mean()
    f1 = cross_val_score(model, x, y, cv=cv, scoring='f1').mean()
    auc = cross_val_score(model, x, y, cv=cv, scoring='roc_auc').mean()

    print(f"{name} CV Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {auc:.4f}")
    print("-" * 35)

### Đánh giá từng model bằng hàm train_test_split

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)
for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    y_prob = model.predict_proba(x_val)[:,1]

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)

    print(f"{name} Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {auc:.4f}")
    print("-" * 35)

### Chọn mô hình tốt nhất

In [ ]:
best_model = models['SVM']
best_model.fit(x, y)

## Xuất ra file kết quả

In [ ]:
# Prepare submission (predict on test)
try:
    preds = best_model.predict(x_test_final)
    submission = pd.DataFrame({
        'PassengerId': test_df['PassengerId'], 
        'Survived': preds.astype(int)
    })
    submission.to_csv('submission.csv', index=False)
    print("File 'submission.csv' created.")
except Exception as e:
    print('Không thể tạo submission tự động:', e)


In [ ]:
# !jupyter nbconvert --to script titanic_test.ipynb

In [ ]:
import pandas as pd

submission = pd.read_csv('/kaggle/working/submission.csv')

print(submission.shape)
print(submission.columns.tolist())
print(submission.head())